# Shuffling

**Shuffling** - The process of redistributing data across partitions so records with the same key end up in the same partition.

- Shuffling is one of the most expensive operations in Spark as it involves moving data across the network between worker nodes.
- The significant overhead costs are necessary for many other operations to perform correctly.

---

# Narrow vs. Wide Transformations

**Narrow vs. Wide Transformations** - Narrow vs. wide just indicates whether or not Spark forces a shuffle. A good way to identify the type of transformation is to examine the dependency between the parent partitions and the resulting partitions.

## Narrow Transformations

A transformation is **narrow** when each output partition depends on only one parent partition. Since each partition can be processed independently, **no shuffle is required**.

| Function | Explanation |
|----------|-------------|
| **`map()`** | Applies a function to every element, producing exactly one output for each input. The partition structure remains unchanged. |
| **`flatMap()`** | Similar to `map()`, but each input can produce zero, one, or many output records. The partition structure still remains unchanged, so no shuffle is needed. |
| **`filter()`** | Filter only performs a check across the data and then drops data if it is not necessary. The partition layout remains the same. |
| **`coalesce()`*** | Reduces the number of partitions by merging existing partitions. By default `shuffle=False`, meaning that it avoids a full shuffle and is more efficient than `repartition()`. |
| **`union()`** | This just simply combines 2 data sets without touching the partition boundaries. |

> **\*** By default, `coalesce()` is a narrow transformation because `shuffle=False`. If `shuffle=True` is specified, it becomes a wide transformation.

---

## Wide Transformations

A transformation is **wide** when an output partition depends on data from multiple parent partitions.

These operations require Spark to **shuffle data** so that related records end up together before computation can continue.

| Transformation Type | Explanation |
|---------------------|-------------|
| **Aggregations and Groupings** (`groupBy`, `reduceByKey`, `aggregateByKey`) | Records with the same key are shuffled to the same partition so Spark can compute the aggregation. The work is still distributed across multiple workers. |
| **Join and Set Operations** | Spark redistributes both datasets so matching keys end up in the same partition before performing the join or set operation. |
| **Partitioning and Reshaping** (`repartition`) | Since the partition layout itself changes, Spark redistributes data across the cluster, requiring a shuffle. |
| **Sorting / Ordering** (`sortBy`, `orderBy`) | Spark redistributes records so each partition contains the correct range of sorted values, allowing the entire dataset to be globally sorted. |

---

# ⭐ THE GOLDEN RULE

> **If Spark can perform the transformation using only the data already inside each partition, it is a narrow transformation. If Spark must move data between partitions before continuing, it is a wide transformation.**

---

# Some Nifty Functions

| Function | Purpose | Shuffle | Narrow / Wide |
|----------|---------|:-------:|:-------------:|
| **`repartition()`** | Redistributes data to create a new number of partitions, typically balancing data evenly across the cluster. | ✅ Yes | **Wide** |
| **`partitionBy()`** | Partitions a Pair RDD using a key so that records with the same key are stored in the same partition. Useful before joins or aggregations. | ✅ Yes | **Wide** |
| **`coalesce()`** | Reduces the number of partitions by merging existing partitions while avoiding a full shuffle by default. Best used for decreasing partitions. | ❌ No* | **Narrow*** |

> **\*** `coalesce()` only avoids a shuffle by default (`shuffle=False`). Setting `shuffle=True` makes it perform a shuffle, turning it into a wide transformation.

---

# Parquet Files

**Parquet Files** - A Parquet file is a **columnar storage format** designed for big data processing.

Instead of storing data **row by row**, it stores data **column by column**.

- CSV files store data in a way that is easy for a human to read.
- A Parquet file stores data in a way that is fast and efficient for Spark to read.
- This is especially handy in large data sets when you may only want to look at one column of a very large table.

In [23]:
#Creating an RDD
from pyspark import SparkContext, SparkConf


conf = SparkConf().setMaster('local[*]').setAppName('WordCount')
sc = SparkContext.getOrCreate(conf)
numbers = sc.parallelize([1,2,3,4,5], 2)

print("RDD:", numbers.collect())
print("Partitions:", numbers.getNumPartitions())

RDD: [1, 2, 3, 4, 5]
Partitions: 2


In [24]:
# Map()

rdd1 = sc.parallelize([1,2,3])
rdd2 = sc.parallelize([4,5,6])

combined = rdd1.union(rdd2)
print(combined.collect())




[1, 2, 3, 4, 5, 6]


In [25]:
# Looking at Partitions
numbers = sc.parallelize(range(20), 4)

print("Partitions:")
for x in numbers.glom().collect():
    print(x)

Partitions:
[0, 1, 2, 3, 4]
[5, 6, 7, 8, 9]
[10, 11, 12, 13, 14]
[15, 16, 17, 18, 19]


In [ ]:
# Coalesce
numbers = sc.parallelize(range(20), 4)
print("Before:", numbers.getNumPartitions())

print("Partitions")
for x in numbers.glom().collect():
    print(x)

smaller = numbers.coalesce(2)
print("\nAfter:", smaller.getNumPartitions())

print("Partitions")
for x in smaller.glom().collect():
    print(x)

larger = numbers.coalesce(8, shuffle=True)

print("\nAfter shuffle")
for x in larger.glom().collect():
    print(x)

Before: 4
Partitions
[0, 1, 2, 3, 4]
[5, 6, 7, 8, 9]
[10, 11, 12, 13, 14]
[15, 16, 17, 18, 19]

After: <bound method RDD.getNumPartitions of CoalescedRDD[196] at coalesce at NativeMethodAccessorImpl.java:0>
Partitions
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

After shuffle
[10, 11, 12, 13, 14]
[]
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[]
[]
[]
[15, 16, 17, 18, 19]
[]


In [35]:
# partitionBy()
pairs = sc.parallelize([
    ("A", 1),
    ("B", 2),
    ("A", 3),
    ("B", 4),
    ("C", 5)
])

partitioned = pairs.partitionBy(3)
for x in partitioned.glom().collect():
    print(x)



[]
[('A', 1), ('A', 3), ('C', 5)]
[('B', 2), ('B', 4)]


In [28]:
# reduceByKey
pairs = sc.parallelize([
    ("A", 1),
    ("B", 2),
    ("A", 3),
    ("B", 4),
    ("A", 5)
])

totals = pairs.reduceByKey(lambda x, y: x + y)

for x in totals.collect():
    print(x)

('B', 6)
('A', 9)


In [29]:
# group by key
pairs = sc.parallelize([
    ("A", 1),
    ("A", 2),
    ("B", 3),
    ("B", 4)
])

grouped = pairs.groupByKey()

for x in grouped.mapValues(list).collect():
    print(x)

('B', [3, 4])
('A', [1, 2])


In [30]:
# sortBy()

numbers = sc.parallelize([8, 3, 1, 7, 2, 6, 5, 4], 2)

print("Before sort:")
for x in numbers.glom().collect():
    print(x, '\n')

sorted_numbers = numbers.sortBy(lambda x: x)

print("\nAfter sort:")
for x in sorted_numbers.glom().collect():
    print(x)

print("\nFinal Output:")
for x in sorted_numbers.collect():
    print(x)

Before sort:
[8, 3, 1, 7] 

[2, 6, 5, 4] 


After sort:
[1, 2, 3, 4, 5]
[6, 7, 8]

Final Output:
1
2
3
4
5
6
7
8


In [31]:
# join
employees = sc.parallelize([
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie"),
    (4, "David")
], 2)

departments = sc.parallelize([
    (1, "Engineering"),
    (2, "HR"),
    (3, "Finance"),
    (4, "Marketing")
], 2)

print("\nEmployee Partitions:")
for x in employees.glom().collect():
    print(x)

print("\nDepartment Partitions:")
for x in departments.glom().collect():
    print(x)

joined = employees.join(departments, numPartitions=2)

print("\nJoined Partitions:")
for x in joined.glom().collect():
    print(x)

print("\nJoined Output:")
for x in joined.collect():
    print(x)



Employee Partitions:
[(1, 'Alice'), (2, 'Bob')]
[(3, 'Charlie'), (4, 'David')]

Department Partitions:
[(1, 'Engineering'), (2, 'HR')]
[(3, 'Finance'), (4, 'Marketing')]

Joined Partitions:
[(2, ('Bob', 'HR')), (4, ('David', 'Marketing'))]
[(1, ('Alice', 'Engineering')), (3, ('Charlie', 'Finance'))]

Joined Output:
(2, ('Bob', 'HR'))
(4, ('David', 'Marketing'))
(1, ('Alice', 'Engineering'))
(3, ('Charlie', 'Finance'))
